In [1]:
import os
import json
import numpy as np
from scipy.stats import t, norm

In [2]:
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n >= 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n >= 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [3]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_None_T4500_C4208/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 10, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.01 ± 0.01',
    'ICU': {'espera': '0.86 ± 0.05', 'tratamiento': '69.93 ± 0.64'},
    'OR': {'espera': '0.5 ± 0.03', 'tratamiento': '12.95 ± 0.04'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '169.85 ± 0.59'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.02 ± 0.04',
    'ICU': {'espera': '0.61 ± 0.05', 'tratamiento': '70.5 ± 0.59'},
    'OR': {'espera': '0.39 ± 0.03', 'tratamiento': '12.98 ± 0.04'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.53 ± 0.72'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.02 ± 0.04',
    'ICU': {'espera': '0.41 ± 0.05', 'tratamiento': '67.29 ± 0.33'},
    'OR': {'espera': '0.22 ± 0.02', 'tratamiento': '13.11 ± 0.03'},
    'SDU_WARD': {'espera': '0.0 ± 0.0', 'tratamiento': '160.54 ± 0.47'}}},
  'promedio_por_hospital': {'Hospital_1': '228.77 ± 0.89',

In [6]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloA_None_T4500_C4208/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 22, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.8 ± 0.04', 'tratamiento': '70.38 ± 0.32'},
    'OR': {'espera': '0.5 ± 0.02', 'tratamiento': '13.0 ± 0.03'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '169.86 ± 0.23'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.56 ± 0.02', 'tratamiento': '70.92 ± 0.27'},
    'OR': {'espera': '0.39 ± 0.01', 'tratamiento': '13.05 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.64 ± 0.29'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.5 ± 0.03', 'tratamiento': '66.48 ± 0.2'},
    'OR': {'espera': '0.26 ± 0.02', 'tratamiento': '12.98 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '159.87 ± 0.19'}}},
  'promedio_por_hospital': {'Hospital_1': '229.47 ± 0.45',
   'Hos

In [5]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados sensibilidad/ModeloA_None_T4500_C4208_H1_SDU_WARD_+1/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 22, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.77 ± 0.03', 'tratamiento': '70.61 ± 0.28'},
    'OR': {'espera': '0.49 ± 0.02', 'tratamiento': '13.01 ± 0.03'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '169.96 ± 0.29'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.54 ± 0.02', 'tratamiento': '71.02 ± 0.24'},
    'OR': {'espera': '0.37 ± 0.02', 'tratamiento': '13.07 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.58 ± 0.29'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.48 ± 0.03', 'tratamiento': '66.54 ± 0.19'},
    'OR': {'espera': '0.24 ± 0.02', 'tratamiento': '12.99 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '159.79 ± 0.24'}}},
  'promedio_por_hospital': {'Hospital_1': '229.89 ± 0.45',
  